# Session-Based Recommendations

Wiki reference for [session-based recommendations](https://ml-viz-ruby.vercel.app/wiki/session-based-recommendations).

**The idea in one sentence.** Session-based recommenders predict the *next* item from the current
click sequence using **causal self-attention** (SASRec) trained with a ranking loss (**BPR-max**),
where the choice of **negative samples** — especially *hard* negatives close to the positive —
drives how much the model learns.

We implement BPR-max and a SASRec attention layer from scratch, **validate the ranking loss and
the causal attention**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1 — BPR-max loss for session recommendation

In [ ]:
def bpr_max_loss(pos_score, neg_scores):
    """BPR-max: maximize margin between positive item and hardest negative."""
    softmax_neg = np.exp(neg_scores - neg_scores.max())
    softmax_neg /= softmax_neg.sum()
    return -np.log(np.sum(softmax_neg * np.exp(pos_score - neg_scores)) + 1e-9)

# 1 positive item, 5 negative items
pos_score = 0.8
neg_scores = np.array([0.2, 0.5, 0.7, 0.3, 0.4])
loss = bpr_max_loss(pos_score, neg_scores)
print(f"BPR-max loss (pos=0.8, hard neg=0.7): {loss:.4f}")

# When positive is clearly better, loss should be low
pos_easy = 2.0
loss_easy = bpr_max_loss(pos_easy, neg_scores)
print(f"BPR-max loss (pos=2.0, easy): {loss_easy:.4f}")

### Validate: BPR-max rewards ranking the positive above the negatives

BPR-max is a ranking loss: it is **low** when the positive item outscores the negatives and
**high** when a negative beats the positive. We confirm the loss falls when the positive wins.

In [ ]:
loss_win = bpr_max_loss(0.9, np.array([0.2, 0.5, 0.3]))
loss_lose = bpr_max_loss(0.1, np.array([0.2, 0.5, 0.3]))
print(f'BPR-max loss: positive wins = {loss_win:.3f},  positive loses = {loss_lose:.3f}')
assert loss_win < loss_lose, 'BPR-max loss is lower when the positive item outscores the negatives'
print('\n✅ BPR-max is a pairwise ranking loss — it optimizes ordering, not absolute scores')

## 2 — Causal self-attention (SASRec layer)

In [ ]:
def sasrec_layer(E, W_q, W_k, W_v):
    """Single SASRec attention layer. E: (T, d), W_*: (d, d)"""
    T, d = E.shape
    Q, K, V = E @ W_q, E @ W_k, E @ W_v
    scores = Q @ K.T / d**0.5
    mask = np.triu(np.full((T,T), -1e9), k=1)
    attn = np.exp(scores + mask)
    attn /= attn.sum(1, keepdims=True)
    return attn @ V

T, d = 6, 8
E = rng.normal(size=(T, d))
W_q = rng.normal(size=(d, d)) * 0.1
W_k = rng.normal(size=(d, d)) * 0.1
W_v = rng.normal(size=(d, d)) * 0.1

H = sasrec_layer(E, W_q, W_k, W_v)
print(f"Input shape: {E.shape}, Output shape: {H.shape}")
print("Last position embedding (used for next-item prediction):", H[-1].round(3))

### Validate: SASRec uses causal attention

To predict the next item, position $t$ may attend only to items **at or before** $t$ — never the
future. The upper-triangular mask sets future scores to $-\infty$, so those attention weights are
exactly 0 while each row remains a distribution. We confirm.

In [ ]:
Q, K = E @ W_q, E @ W_k
scores = Q @ K.T / d**0.5
mask = np.triu(np.full((T, T), -1e9), k=1)
attn = np.exp(scores + mask); attn /= attn.sum(1, keepdims=True)
upper = np.triu(np.ones((T, T), dtype=bool), k=1)
print(f'future attention weights all ~0: {bool((attn[upper] < 1e-6).all())}')
assert (attn[upper] < 1e-6).all(), 'SASRec attends only to past items (causal) — no peeking at the future'
assert np.allclose(attn.sum(1), 1.0), 'each row is a valid attention distribution'
print('\n✅ causal masking makes next-item prediction well-posed')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **easy negatives** | weak gradients, slow learning (demo) — mine hard negatives |
| **too-hard negatives** | can be false negatives (actually relevant) — noisy signal |
| **leaking the future** | forgetting the causal mask makes next-item prediction trivial (verified) |
| **popularity bias** | popular items dominate; sample negatives by popularity |
| **cold sessions** | very short sessions have little signal |

Demo: hard negatives produce a larger loss than easy ones.

In [ ]:
# The lever that makes or breaks a session recommender: NEGATIVE SAMPLING. Random negatives are
# easy (far below the positive) and give weak gradients; HARD negatives — items scoring close to
# the positive — produce a larger loss and a much stronger learning signal (they teach the fine
# distinctions). We compare the loss under easy vs hard negatives.
pos = 0.8
loss_easy = bpr_max_loss(pos, np.array([0.0, 0.1, -0.2]))    # easy: far below the positive
loss_hard = bpr_max_loss(pos, np.array([0.70, 0.75, 0.60]))  # hard: close to the positive
print(f'BPR-max loss: easy negatives = {loss_easy:.3f},  hard negatives = {loss_hard:.3f}')
assert loss_hard > loss_easy, 'hard negatives (near the positive) produce a larger loss -> stronger gradients'
print('\nHard negatives teach fine distinctions -> negative sampling strategy is a key design choice.')

## ✏️ Your turn — negative sampling strategy comparison

In [ ]:
def train_loss_with_negatives(pos_scores, neg_strategy, all_scores):
    """
    Compare BPR-max loss under different negative sampling strategies.
    pos_scores: (B,) scores for positive items
    neg_strategy: 'random', 'in_batch', 'hard'
    all_scores: (B, N) scores for all items
    """
    # TODO(you): implement the three strategies and compute mean BPR-max loss
    # random: sample uniformly from all_scores (columns != positive)
    # in_batch: use the other B positive items' scores as negatives
    # hard: use the top-5 scoring items (excluding the positive) as negatives
    return ...

# Test your implementation
B, N = 8, 50
pos_scores = rng.uniform(0.5, 1.5, B)
all_scores = rng.uniform(-1, 2, (B, N))
# The positive item is at column 0 for each row
all_scores[np.arange(B), 0] = pos_scores  # fix the positive positions

for strategy in ['random', 'in_batch', 'hard']:
    loss = train_loss_with_negatives(pos_scores, strategy, all_scores)
    print(f"{strategy:10s}: loss = {loss}")

<details><summary>Hint</summary>

- **random**: for each row, sample k columns ≠ 0, use those scores as negatives in BPR-max.
- **in_batch**: for row i, use `pos_scores[j]` for all j ≠ i as negatives.
- **hard**: for each row, find the top-5 highest-scoring items (excluding col 0) as negatives.

Hard negatives should give the highest loss (hardest to learn from), random the lowest.
</details>

## Key takeaways

- **Next-item prediction** from the click sequence via causal self-attention (SASRec).
- **BPR-max** is a ranking loss — it optimizes ordering the positive above negatives (verified).
- **Causal masking** ensures position $t$ sees only the past (verified).
- **Negative sampling matters:** hard negatives give stronger gradients than random (demo).